# DATASET


In [1]:
from urllib.request import urlopen
import gzip

url = "https://snap.stanford.edu/data/roadNet-CA.txt.gz"
with urlopen(url) as response:
    dataset_lines = gzip.decompress(response.read()).decode().splitlines()

print("Number of nodes:", int(dataset_lines[2].split()[2]))
print("Sample connections (FromNodeId -> ToNodeId):")
for line in dataset_lines[4:14]:
    print(line)


Number of nodes: 1965206
Sample connections (FromNodeId -> ToNodeId):
0	1
0	2
0	469
1	0
1	6
1	385
2	0
2	3
469	0
469	380


# DATA PREPROCESSING


In [2]:
graph = {}
invalid_rows = 0

for line in dataset_lines:
    if not line.strip() or line.lstrip().startswith("#"):
        continue
    try:
        first_node, second_node = map(int, line.split())
        if first_node < 0 or second_node < 0:
            raise ValueError
    except ValueError:
        invalid_rows += 1
        continue

    # Every endpoint becomes a valid node; IDs need not be consecutive.
    if first_node not in graph:
        graph[first_node] = []
    if second_node not in graph:
        graph[second_node] = []

    # Add both directions once, including when reverse rows already exist.
    if second_node not in graph[first_node]:
        graph[first_node].append(second_node)
    if first_node not in graph[second_node]:
        graph[second_node].append(first_node)

if not graph:
    raise ValueError("The dataset contains no valid connections.")

edge_count = sum(
    1 for node in graph for neighbor in graph[node] if node <= neighbor
)
print("PREPROCESSING")
print(f"Valid nodes: {len(graph):,}")
print(f"Undirected edges: {edge_count:,}")
print(f"Invalid rows skipped: {invalid_rows:,}")
print("All connections lead to existing nodes.")


PREPROCESSING
Valid nodes: 1,965,206
Undirected edges: 2,766,607
Invalid rows skipped: 0
All connections lead to existing nodes.


# ITERATIVE DEEPENING SEARCH (IDS) IMPLEMENTATION

Every edge has cost 1. The first path returned by this algorithm is already a shortest path. The output reports the same measured run under both objectives; there is no second search phase.


In [3]:
# Search increasing depths. The first returned path has the fewest edges.
from time import perf_counter

def iterative_deepening_search(graph, start_node, goal_node,
                             maximum_depth=20, time_limit=10):
    result = {"path": None, "nodes_expanded": 0,
              "status": "Depth limit reached", "depth_limit": None}
    if start_node not in graph or goal_node not in graph:
        result["status"] = "Invalid start or goal node"
        return result

    search_start = perf_counter()

    def depth_limited_search(current_node, remaining_depth, path):
        # Stop long searches before trying deeper paths.
        if perf_counter() - search_start >= time_limit:
            result["status"] = "Time limit reached"
            return None
        if current_node == goal_node:
            return path.copy()
        if remaining_depth == 0:
            return None

        # Count an expansion when this node's neighbors are explored.
        result["nodes_expanded"] += 1
        for neighbor in graph[current_node]:
            if neighbor not in path:  # Avoid cycles in the current path.
                path.append(neighbor)
                found_path = depth_limited_search(
                    neighbor, remaining_depth - 1, path
                )
                path.pop()  # Backtrack before exploring another branch.
                if found_path is not None:
                    return found_path
                if result["status"] == "Time limit reached":
                    return None
        return None

    for depth_limit in range(maximum_depth + 1):
        result["depth_limit"] = depth_limit
        found_path = depth_limited_search(start_node, depth_limit, [start_node])
        if found_path is not None:
            result["path"] = found_path
            result["status"] = "Path found"
            break
        if result["status"] == "Time limit reached":
            break
    return result


# Change these values to try another route. Start with nearby nodes for IDS.
start_node = 0
goal_node = 5
maximum_depth = 20  # Keep this small because IDS uses recursive depth search.
time_limit = 10  # Seconds allowed for each algorithm.

run_start = perf_counter()
ids_result = iterative_deepening_search(
    graph, start_node, goal_node, maximum_depth, time_limit
)
ids_time = perf_counter() - run_start

print("=" * 60)
print("ITERATIVE DEEPENING SEARCH (IDS)")
print("=" * 60)
print("Start node:", start_node)
print("Goal node:", goal_node)
print("Status:", ids_result["status"])
print("Allowed depth:", maximum_depth)
print("Last depth limit tried:", ids_result["depth_limit"])
print("Time limit (seconds):", time_limit)
print(f"Total search time (seconds): {ids_time:.9f}")
print("Node expansions (including repeats):", ids_result["nodes_expanded"])
if ids_result["path"] is not None:
    path_with_arrows = " → ".join(str(node) for node in ids_result["path"])
    print("First path found:", path_with_arrows)
    print("Shortest path:", path_with_arrows)
    print("Shortest path length (edges):", len(ids_result["path"]) - 1)
    print(f"Time to any path (seconds): {ids_time:.9f}")
    print(f"Time to shortest path (seconds): {ids_time:.9f}")
    print("IDS finds the shallowest solution first, so both times are the same.")
else:
    print("First path / shortest path: Not found within the stated limits.")
print("Each edge costs 1; shortest means fewest edges, not road distance.")


ITERATIVE DEEPENING SEARCH (IDS)
Start node: 0
Goal node: 5
Status: Path found
Allowed depth: 20
Last depth limit tried: 3
Time limit (seconds): 10
Total search time (seconds): 0.000030500
Node expansions (including repeats): 8
First path found: 0 → 1 → 6 → 5
Shortest path: 0 → 1 → 6 → 5
Shortest path length (edges): 3
Time to any path (seconds): 0.000030500
Time to shortest path (seconds): 0.000030500
IDS finds the shallowest solution first, so both times are the same.
Each edge costs 1; shortest means fewest edges, not road distance.


# BIDIRECTIONAL BFS IMPLEMENTATION

Every edge has cost 1. The first path returned by this algorithm is already a shortest path. The output reports the same measured run under both objectives; there is no second search phase.


In [4]:
from collections import deque

def bidirectional_bfs(graph, start_node, goal_node, time_limit=10):
    result = {"path": None, "nodes_expanded": 0, "status": "No path exists"}
    if start_node not in graph or goal_node not in graph:
        result["status"] = "Invalid start or goal node"
        return result
    if start_node == goal_node:
        result["path"] = [start_node]
        result["status"] = "Path found"
        return result

    start_frontier = deque([start_node])
    goal_frontier = deque([goal_node])
    # Parent dictionaries also record which nodes each side has visited.
    start_parents = {start_node: None}
    goal_parents = {goal_node: None}
    meeting_node = None
    search_start = perf_counter()

    while start_frontier and goal_frontier:
        # Explore one full BFS layer from the smaller frontier.
        if len(start_frontier) <= len(goal_frontier):
            frontier = start_frontier
            parents = start_parents
            other_parents = goal_parents
        else:
            frontier = goal_frontier
            parents = goal_parents
            other_parents = start_parents

        for step in range(len(frontier)):
            if perf_counter() - search_start >= time_limit:
                result["status"] = "Time limit reached"
                break
            current_node = frontier.popleft()
            result["nodes_expanded"] += 1
            for neighbor in graph[current_node]:
                if neighbor in parents:
                    continue
                parents[neighbor] = current_node
                frontier.append(neighbor)
                if neighbor in other_parents:
                    meeting_node = neighbor
                    break
            if meeting_node is not None:
                break
        if meeting_node is not None or result["status"] == "Time limit reached":
            break

    if meeting_node is not None:
        # Follow parents from the meeting point back to the start.
        path = []
        current_node = meeting_node
        while current_node is not None:
            path.append(current_node)
            current_node = start_parents[current_node]
        path.reverse()

        # Add the remaining part from the meeting point to the goal.
        current_node = goal_parents[meeting_node]
        while current_node is not None:
            path.append(current_node)
            current_node = goal_parents[current_node]
        result["path"] = path
        result["status"] = "Path found"
    return result


# Reuse the same start node, goal node, and time limit as IDS.
run_start = perf_counter()
bfs_result = bidirectional_bfs(graph, start_node, goal_node, time_limit)
bfs_time = perf_counter() - run_start

print("=" * 60)
print("BIDIRECTIONAL BREADTH-FIRST SEARCH (BFS)")
print("=" * 60)
print("Start node:", start_node)
print("Goal node:", goal_node)
print("Status:", bfs_result["status"])
print("Time limit (seconds):", time_limit)
print(f"Total search time (seconds): {bfs_time:.9f}")
print("Node expansions (both directions):", bfs_result["nodes_expanded"])
if bfs_result["path"] is not None:
    path_with_arrows = " → ".join(str(node) for node in bfs_result["path"])
    print("First path found:", path_with_arrows)
    print("Shortest path:", path_with_arrows)
    print("Shortest path length (edges):", len(bfs_result["path"]) - 1)
    print(f"Time to any path (seconds): {bfs_time:.9f}")
    print(f"Time to shortest path (seconds): {bfs_time:.9f}")
    print("BFS layers make the first returned path shortest, so both times are the same.")
else:
    print("First path / shortest path: None returned; check the status above.")
print("Each edge costs 1; shortest means fewest edges, not road distance.")

# Compare the complete returned paths; downloading and printing are not timed.
print("\nANY PATH AND SHORTEST PATH: COMPARISON")
print(f"{'Algorithm':<20} {'Any path (s)':>14} {'Shortest (s)':>14} {'Edges':>8} {'Expanded':>10}")
print("-" * 70)
for name, result, search_time in [
    ("IDS", ids_result, ids_time),
    ("Bidirectional BFS", bfs_result, bfs_time),
]:
    if result["path"] is not None:
        measured_time = f"{search_time:.9f}"
        path_length = str(len(result["path"]) - 1)
    else:
        measured_time = "Not found"
        path_length = "N/A"
    print(f"{name:<20} {measured_time:>14} {measured_time:>14} {path_length:>8} {result['nodes_expanded']:>10}")

print("Both timing columns use the same run: the first solution is already shortest.")
if ids_result["path"] is not None and bfs_result["path"] is not None:
    print("Equal path lengths:", len(ids_result["path"]) == len(bfs_result["path"]))
    if ids_time < bfs_time:
        print("Lower measured time for both objectives in this run: IDS")
    elif bfs_time < ids_time:
        print("Lower measured time for both objectives in this run: Bidirectional BFS")
    else:
        print("Both algorithms recorded the same time in this run.")
else:
    print("Compare completion status above; both algorithms did not return a path.")
print("These are single-run timings; repeat queries before drawing general conclusions.")


BIDIRECTIONAL BREADTH-FIRST SEARCH (BFS)
Start node: 0
Goal node: 5
Status: Path found
Time limit (seconds): 10
Total search time (seconds): 0.000015900
Node expansions (both directions): 3
First path found: 0 → 1 → 6 → 5
Shortest path: 0 → 1 → 6 → 5
Shortest path length (edges): 3
Time to any path (seconds): 0.000015900
Time to shortest path (seconds): 0.000015900
BFS layers make the first returned path shortest, so both times are the same.
Each edge costs 1; shortest means fewest edges, not road distance.

ANY PATH AND SHORTEST PATH: COMPARISON
Algorithm              Any path (s)   Shortest (s)    Edges   Expanded
----------------------------------------------------------------------
IDS                     0.000030500    0.000030500        3          8
Bidirectional BFS       0.000015900    0.000015900        3          3
Both timing columns use the same run: the first solution is already shortest.
Equal path lengths: True
Lower measured time for both objectives in this run: Bidirec